In [1]:
# Install required libraries for Kaggle
# !pip install boto3 sentence-transformers transformers requests -q

import os
import glob
import json
import requests
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import pandas as pd
import numpy as np
from collections import Counter
import re
import warnings

# Suppress verbose HuggingFace warnings for a cleaner output
warnings.filterwarnings("ignore")

# Kaggle-specific working directories
WORK_DIR = '/kaggle/working/'
DATA_DIR = os.path.join(WORK_DIR, 'pmc_articles')

# Teardown phase: Destroy the directory if it exists from a previous run
if os.path.exists(DATA_DIR):
    print("Purging old storage...")
    shutil.rmtree(DATA_DIR)
    
os.makedirs(DATA_DIR, exist_ok=True)

# Configure anonymous access to S3
s3_client = boto3.client(
    's3', 
    region_name='us-east-1', 
    config=Config(signature_version=UNSIGNED)
)
BUCKET_NAME = 'pmc-oa-opendata'

print("Dependencies installed and environment configured.")

Dependencies installed and environment configured.


In [2]:
def fetch_pmcids_from_api(query, max_results=5):
    """Tier 1: Uses the NCBI API to find relevant PMCIDs without downloading text."""
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {"db": "pmc", "term": query, "retmode": "json", "retmax": max_results}
    
    response = requests.get(base_url, params=params)
    if response.status_code == 200:
        pmid_list = response.json().get("esearchresult", {}).get("idlist", [])
        return [f"PMC{pmid}" for pmid in pmid_list]
    return []

def download_articles(pmcid_list):
    """Tier 2: Downloads only the highly relevant articles from S3."""
    downloaded_paths = []
    for pmcid in pmcid_list:
        s3_key = f"deprecated/oa_comm/txt/all/{pmcid}.txt" 
        local_path = os.path.join(DATA_DIR, f"{pmcid}.txt")
        
        if not os.path.exists(local_path):
            try:
                response = s3_client.get_object(Bucket=BUCKET_NAME, Key=s3_key)
                with open(local_path, 'wb') as f:
                    f.write(response['Body'].read())
                downloaded_paths.append(local_path)
            except Exception:
                pass # Skip silently if file is missing in S3
        else:
            downloaded_paths.append(local_path)
    return downloaded_paths

def clean_full_text(text):
    """Cleans the text and aggressively filters out academic boilerplate."""
    paragraphs = text.split('\n\n')
    valid_paragraphs = []
    
    # The Boilerplate Blacklist
    blacklist = [
        "creative commons", "copyright", "conflict of interest", 
        "competing interest", "correspondence", "author contribution",
        "none declared", "plos policies", "publisher's note",
        "peer review", "open access", "submitted to", "funding"
    ]
    
    for p in paragraphs:
        clean_p = " ".join(p.split())
        contains_boilerplate = any(bad in clean_p.lower() for bad in blacklist)
        
        if len(clean_p) > 200 and not contains_boilerplate:
            valid_paragraphs.append(clean_p)
            
    return valid_paragraphs

print("Data Ingestion functions are ready.")

Data Ingestion functions are ready.


In [3]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

print("1/3 Loading Retriever/Extractor Model (all-MiniLM)...")
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2') 

print("2/3 Loading Summarizer Model (facebook/bart-large-cnn)...")
summarizer_name = "facebook/bart-large-cnn"
sum_tokenizer = AutoTokenizer.from_pretrained(summarizer_name)
sum_model = AutoModelForSeq2SeqLM.from_pretrained(summarizer_name)

print("3/3 Loading Verifier Model (distilbert zero-shot)...")
verifier = pipeline("zero-shot-classification", model="typeform/distilbert-base-uncased-mnli", device=-1)

print("\nAll AI Models successfully loaded into memory.")

1/3 Loading Retriever/Extractor Model (all-MiniLM)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/258 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]


All AI Models successfully loaded into memory.


In [4]:
from sklearn.metrics.pairwise import cosine_similarity

def extractor_agent(query, document_dicts):
    """Embeds paragraphs dynamically and extracts all chunks above the relevance threshold."""
    query_embedding = embedder.encode([query])
    extracted_docs = []
    
    for doc in document_dicts:
        if not doc["paragraphs"]: continue
            
        para_embeddings = embedder.encode(doc["paragraphs"])
        similarities = cosine_similarity(query_embedding, para_embeddings)[0]
        
        # Find the single best paragraph matching the query within this document
        best_idx = np.argmax(similarities)
        best_score = similarities[best_idx]
        
        # NEW REQUIREMENT: Only keep it if the resemblance is over 40% (0.40)
        if best_score >= 0.40:
            extracted_docs.append({
                "pmcid": doc["pmcid"],
                "relevance_score": best_score,
                "text": doc["paragraphs"][best_idx]
            })
            
    # Sort by relevance, highest to lowest
    extracted_docs = sorted(extracted_docs, key=lambda x: x["relevance_score"], reverse=True)
    
    # NEW REQUIREMENT: Return all matching documents, do not limit to top 2
    return extracted_docs

def extract_keywords(text, num_keywords=5):
    """Term-frequency keyword extraction."""
    words = re.findall(r'\b[a-z]{4,}\b', text.lower())
    stop_words = {'that', 'with', 'from', 'this', 'were', 'which', 'their', 'have', 'been', 'will', 'also', 'than', 'these'}
    filtered_words = [w for w in words if w not in stop_words]
    most_common = Counter(filtered_words).most_common(num_keywords)
    return [word for word, count in most_common]

def summarizer_agent(extracted_docs):
    """Generates concise scientific summaries using BART."""
    summarized_results = []
    for doc in extracted_docs:
        input_text = doc["text"][:1024] 
        inputs = sum_tokenizer(input_text, return_tensors="pt", max_length=1024, truncation=True)
        
        summary_ids = sum_model.generate(
            inputs.input_ids, max_length=75, min_length=25, 
            length_penalty=2.0, num_beams=4, early_stopping=True
        )
        
        summary_text = sum_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        keywords = extract_keywords(doc["text"])
        
        doc["summary"] = summary_text
        doc["keywords"] = keywords
        summarized_results.append(doc)
        
    return summarized_results

def verifier_agent(summarized_docs):
    """Categorizes the findings into target themes."""
    THEMES = ["Deep Learning", "Clinical Trial", "Traditional Methods"]
    verified_results = []
    
    for doc in summarized_docs:
        result = verifier(doc["summary"], candidate_labels=THEMES)
        doc["theme"] = result['labels'][0]
        doc["theme_confidence"] = result['scores'][0]
        verified_results.append(doc)
        
    return verified_results

print("Multi-Agent logic updated with new thresholds.")

Multi-Agent logic updated with new thresholds.


In [5]:
def run_case_study_pipeline(query):
    print(f"\n{'='*60}\nQUERY: '{query}'\n{'='*60}")
    
    # 1. API Fetch & Download
    # NEW REQUIREMENT: Widened the net to 30 to bypass missing S3 files
    print("1. [Data Tier] Querying NCBI API & Downloading S3 files...")
    target_pmcids = fetch_pmcids_from_api(query, max_results=30) 
    file_paths = download_articles(target_pmcids)
    
    print(f"   -> Successfully downloaded {len(file_paths)} files from S3.")
    
    # Parse downloaded files
    documents = []
    for path in file_paths:
        with open(path, 'r', encoding='utf-8') as f:
            pmcid = os.path.basename(path).replace('.txt', '')
            paragraphs = clean_full_text(f.read())
            documents.append({"pmcid": pmcid, "paragraphs": paragraphs})
    
    # 2. Extractor
    print("2. [Extractor Agent] Scanning documents for ->40% semantic match...")
    extracted_docs = extractor_agent(query, documents)
    
    if not extracted_docs:
        print("No passages met the 40% relevance threshold.")
        return
    
    print(f"   -> Found {len(extracted_docs)} highly relevant passages.")
    
    # 3. Summarizer
    print("3. [Summarizer Agent] Compressing passages and generating keywords...")
    summarized_docs = summarizer_agent(extracted_docs)
    
    # 4. Verifier
    print("4. [Verifier Agent] Categorizing outputs against case-study themes...")
    final_results = verifier_agent(summarized_docs)
    
    # Output
    for i, res in enumerate(final_results, 1):
        print(f"\n--- Result {i} (PMCID: {res['pmcid']}) ---")
        print(f"Match Score: {res['relevance_score']:.2f}")
        print(f"Theme: {res['theme']} (Confidence: {res['theme_confidence']:.2f})")
        print(f"Keywords: {', '.join(res['keywords'])}")
        print(f"Summary: {res['summary']}")

print("Pipeline execution logic updated.")

Pipeline execution logic updated.


In [ ]:
# The specific test queries mandated by the case study
test_queries = [
    "Adverse events with mRNA vaccines in pediatrics",
    "Transformer-based models for protein folding",
    "Clinical trial outcomes for monoclonal antibodies in oncology"
]

for q in test_queries:
    run_case_study_pipeline(q)


QUERY: 'Adverse events with mRNA vaccines in pediatrics'
1. [Data Tier] Querying NCBI API & Downloading S3 files...
   -> Successfully downloaded 0 files from S3.
2. [Extractor Agent] Scanning documents for ->40% semantic match...
No passages met the 40% relevance threshold.

QUERY: 'Transformer-based models for protein folding'
1. [Data Tier] Querying NCBI API & Downloading S3 files...
   -> Successfully downloaded 23 files from S3.
2. [Extractor Agent] Scanning documents for ->40% semantic match...
   -> Found 21 highly relevant passages.
3. [Summarizer Agent] Compressing passages and generating keywords...
